## Level 2 Processing 

### L1b (λ) → L2s1(λ) → L2s2(λ) → L2s3(λ)  → L2s4(λ) → L2s5(λ)**→  L2s6(λ)

1) Iπ/F Correction,
2) Statistical Polishing,
3) Thermal Removal,
4) Photometric Correction,
5) Ground Truth Correction (an optional step**) - NOT applied to L2 data in the PDS but calibration files required are archived 
6) Flag degraded channels.
The Level 2 applies the above steps in serial manner:


In [ ]:
from datetime import datetime
import pandas as pd
from pathlib import Path
from typing import Optional
import numpy as np
from astropy.io import fits
from scipy.optimize import brentq  # was fitting this way

### Iπ/F Correction
L2s1(λ) = L1b(λ) * π / (SolarIrrad(λ) / d )\
Where:
- λ is the wavelength (global or target)
- At-Sensor Radiance from Level 1B (version 3.0 in the PDS archive)
- SolarIrrad(λ) is a Global or Target file providing the exo-atmospheric solar
spectrum at 1 Astronomical Unit as determined with MODTRAN. See Anderson
et al., [2000] and Kurucz, [1995].\
o Units are irradiance W/(m2 µm)
- Normalized Sun-Moon Distance, d, from Level 1B\
o Value for the scene mean in units of AU supplied by the
SOLAR_DISTANCE keyword in *L1B.LBL

In [ ]:
def calculate_radiance_factor(image: np.ndarray, sol_spec_path: Path, d: float = 1.0):
    """
    Get I/F (radiance factor) using solar distance and solar spectrum file.
    L2s1(λ) = L1b(λ) * π / (SolarIrrad(λ) / d^2)

    SolarIrrad(λ) is a Global or Target file providing the exo-atmospheric
    solar spectrum at 1 Astronomical Unit as determined with MODTRAN.
    See Anderson et al., [2000] and Kurucz, [1995] (DPSIS 2011)
    """
    import pandas as pd

    image *= np.pi

    solarrad = pd.read_fwf(sol_spec_path, names=['wavelength', 'sol_irrad'])
    
    solarrad['sol_irrad'] = solarrad['sol_irrad'] / (d**2)

    image /= solarrad['sol_irrad'].values[np.newaxis, :, np.newaxis]

    return image

### Statistical Polishing 

L2s2(λ) = L2s1(λ) * gSP(λ) + oSP(λ))\
where\
• λ is the wavelength (global or target)\
• gSP(λ) is a Global or Target File of derived gains (radiance correction factors)
for statistical polishing\
    o Gains are unitless\
    o These values are supplied by the M3 science team. A summary of how
    gSP(λ) was derived is provided here:\
        1) Select a suite of featureless spectra, about several million pixels
            for instrument warm and instrument cold states (colored lines in
            Figure 2-21) found by searching for spectra with weak to no
            spectral features (described in Clark et al., [2012]).\
        2) Hand fit a spline through the spectrum (black line in Figure 2-21).
            The cubic spline is not physically hand drawn, but rather a cubic
            spline is fit through the data at channels where the polisher
            correction factor equals 1.0. \
        3) Divide the average spectrum by a fitted line continuum = S. The
            results are shown in Figure 2-22. \
        4) Radiance Correction factor (RC) = average spectrum/S for the L2
            delivery, this is called U2RC1 (earlier statistical polishing work
            combined cold and warm data and was designation R4RC1). The
            U2RC1 multiplier spectra are shown in Figure 2-22. \
        5) Four correction spectra were derived for instrument warm and
            instrument cold states, each for global and target modes.\
• oSP(λ) is a Global or Target File of derived offsets for statistical polishing\
    o Offsets are unitless\
    o Supplied by the science team\
    o For this PDS delivery, oSP(λ) is set to zero (no offset).

In [ ]:
def apply_stat_polishing(image: np.ndarray, stat_pol_path: Path):
    """
    Multiply by statistical polishing coefficients. These are temperature
    dependent, there is a "warm" and "cold" version determined by the date.
    There used to be an additional statistical polishing offset but for the
    latest delivery I believe it was all 0.

    L2s2(λ) = L2s1(λ) * gSP(λ) + oSP(λ))
    """
    import pandas as pd

    statpol = pd.read_fwf(stat_pol_path, names=['channel', 'wavelength',
                                                 'mult_factor', 'offset'])

    image *= statpol['mult_factor'].values[np.newaxis, :, np.newaxis]

    # image += statpol['offset'].values[np.newaxis, :, np.newaxis]

    return image


### Iterative Thermal Removal 

L2s3(λ) = F(L2s2(λ) )

where F(L2s2(λ) ) is: 
1) Remove only the SOLAR_DISTANCE correction, applied in L2 Step 1, to
the input data L2s2(λ).
2) For each spectrum, the I/F is linearly projected from wavelengths A and B
in Table 1 (below) to wavelength C.
3) The projected I/F at wavelength C is subtracted from the observed I/F at
wavelength C to give the thermal component T1.
4) If the difference is negative, the temperature is not derived. (Private
correspondence from R. Clark: This can happen frequently because
there is a lot of water on the Moon; see Clark et al., [2012].) If the
difference is positive, the difference is assumed to be thermal emission
and the temperature whose black body emission that best matches that of
T1, is derived. The emissivity is assumed to be constant with wavelength
equal to 1- I/F at wavelength A.
5) The derived thermal emission is subtracted from the observed I/F, giving
a new I/F estimate IOF1 as a function of wavelength.
6) R1 is then corrected for incidence angle and phase angle effects = IOF 1c
(as of the writing of this paper, phase angle corrections for the 2 to 3-µm
wavelength region for the lunar surface are not yet available).
7) The wavelength dependent emissivity is then computed from e = 1 –
IOF1c.
8) Next a new projection using IOF 1 is made from wavelengths D and E
(Table 2.6) to a new I/F at wavelength C, and the difference, T2,
computed.
9) A new temperature (T) is derived from the thermal difference (T2) by
computing a new black body that includes the wavelength dependent
emissivity estimate, e, and cosine correction (and once available, a phase
correction).
10) The thermal emission, ro2eBo(T)/Fsun, is computed from this 2nd iteration
estimate and subtracted from the original I/F spectrum, producing a new
estimate, IOF2c.
11) If the derived temperature difference from the first and second iteration is
less 2 K, the solution is complete. Otherwise, a third iteration is
computed by going back to step 6 and substituting IOF2c for IOF1c.
12) Put back SOLAR_DISTANCE correction removed in 1) above.
    
- Tests were made with additional iterations (up to 12) and it was
determined that after 3 iterations, thermal emission was added back into
some spectra that displayed 2-µm pyroxene bands. It was determined
that 2 to 3 iterations provided the optimum solution for a single
temperature.
- When the method cannot detect excess thermal emission in the presence
of sufficiently strong absorption in the 3-µm region in M3 data (defined by
the initial projection of the spectrum discussed above falling above the
observed data), no derivation of a temperature is performed and no
removal of thermal emission in M3 data is performed. This means that
water and hydroxyl bearing areas have no thermal emission removed and
that any mapping of these absorptions and derivations of abundance is
conservative and provides lower limits. By the nature of the algorithm
design, that of linear extrapolation, the algorithm will not produce a
downturn in M3 data, and thus can not introduce an artificial water
absorption when multiple iterations are done in the retrieval of the
reflectance spectrum.
- It is an almost certainty that thermal emission
hides areas containing water in the lunar surface and reduces the water
band strengths in almost all areas measured by M3, except in low sun
angle areas (for example, near the poles and slopes facing away from the
sun at mid-to-high latitudes, where the surface temperatures are below
about 250 K). More on this method can be found in Clark et al., [2011].
- We limit the I/F cosine correction and the computed reflectance to avoid
errors due to lower spatial resolution to topography and registration
topography to prevent very high computed reflectances.
- If cosine(incidence) is less than 0.05 then 0.05 is used as the value for
cosine(incidence). If the computed reflectance is greater than 0.6 then 0.6
is used as the value for computed reflectance.

In [ ]:
# wavelengths
WL_A = 1.55
WL_B = 2.35
WL_C = 2.7
WL_D = 2.28
WL_E = 2.59

# it goes from A and B proj to C 
# and then from C and D proj to C 

COS_I_MIN = 0.05      
REFLECTANCE_MAX = 0.6
TEMP_CONV_THRESH = 2.0  # DPSIS step 11 convergence threshold (K)
MAX_ITER = 3            # max 3 iterations

H = 6.62607015e-34      # J*s
C_LIGHT = 2.99792458e8  # m/s
K_B = 1.380649e-23      # Boltzmann J/K


def nearest_band(wavelengths_um: np.ndarray, target_um: float):
    """index of the band closest to target (in um)."""
    return int(np.argmin(np.abs(wavelengths_um - target_um)))

def linear_project(
    wl: np.ndarray,
    iof: np.ndarray,
    idx_a: int,
    idx_b: int,
    idx_c: int,
) -> float:
    """
    linearly project from band a and b to c. 

    "if the computed reflectance is greater than 0.6 then 0.6 
    is used as the value for computed reflectance." 
    """
    wl_a, wl_b, wl_c = wl[idx_a], wl[idx_b], wl[idx_c]
    iof_a = iof[idx_a]
    iof_b = iof[idx_b]
    slope = (iof_b - iof_a) / (wl_b - wl_a)
    return iof_a + slope * (wl_c - wl_a)

def planck_um(wavelength_um, T):
    wl_m = wavelength_um * 1e-6
    exponent = H * C_LIGHT / (wl_m * K_B * T)
    B = (2 * H * C_LIGHT**2/ (wl_m**5))/  np.expm1(exponent)
    # per um, solar spec is in um 
    return B * 1e-6 

def planck_array(wavelengths_um: np.ndarray, T: float) -> np.ndarray:
    wl_m = wavelengths_um * 1e-6
    exponent = H * C_LIGHT / (wl_m * K_B * T)
    B = (2 * H * C_LIGHT**2 / (wl_m**5)) / np.expm1(exponent)
    return B * 1e-6
    
def fit_temperature(    
    T1: float,
    emissivity: float,
    wavelength_c_um: float,
    solar_spec_at_c: float,
    rs_sq: float,
):
    B_target = T1 * solar_spec_at_c / (rs_sq * emissivity)
    temps = np.arange(200, 600.0, 0.1) # the min temp on the particular obs I'm testing is ~220
                                       # idk the actual range in all m3 
    model = planck_um(wavelength_c_um, temps)
    temperature = temps[np.argmin(np.abs(model - B_target))]
    return temperature 
    
def thermal_emission_spectrum(
    wavelengths_um: np.ndarray,
    temperature_K: float,
    emissivity: np.ndarray,
    solar_spec: np.ndarray,
    rs_sq: float,
):
    """
    thermal(lambda) = rs2 * e(lambda) * Bo(T, lambda) / Fsun(lambda)
    """
    B = planck_array(wavelengths_um, temperature_K)
    return rs_sq * emissivity * B / solar_spec

def remove_thermal_single_spectrum(
    iof_spectrum: np.ndarray,
    solar_spec: np.ndarray,
    cos_incidence: float,
    d: float,
    wl: np.ndarray,
    iA: int,
    iB: int,
    iC: int,
    iD: int,
    iE: int,
) -> tuple[np.ndarray, float, int]:

    iof = iof_spectrum.copy()

    iof_1au = iof_spectrum 
    
    d_sq = d ** 2 # not used atm, I think this actually should all be 1 AU bc we removed the distance
                  # corr in the parent function 
                  # which is why everything following this uses rs_sq = 1.0 

    ### ! STEP NUMBERS ALIGN WITH THE DPSIS NOT CLARK 2011, CLARK 2011 DIDN'T HAVE DPSIS STEP 1 ! ### 
    
    # STEP 2: "For each spectrum, the I/F is linearly projected from wavelengths A and B
    # in Table 1 (below) to wavelength C"
    iof_proj_C1 = linear_project(wl, iof, iA, iB, iC)
    # "If the computed reflectance is greater than 0.6 then 0.6 is used as the value for 
    # computed reflectance." 
    iof_proj_C1 = np.clip(iof_proj_C1, a_min=None, a_max=0.6) 

    # STEP 3: "The projected I/F at wavelength C is subtracted from the observed I/F at
    # wavelength C to give the thermal component T1."
    T1 = iof[iC] - iof_proj_C1

    # STEP 4: T1 < 0: no thermal excess (water/OH absorption)
    # "If the difference is negative, the temperature is not derived. (Private
    # correspondence from R. Clark: This can happen frequently because
    # there is a lot of water on the Moon; see Clark et al., [2012].)"
    if T1 < 0:
        return iof, 0.0, 6

    # STEP 4 cont.: Constant emissivity = 1 - I/F at band A 
    # "If the difference is positive, the difference is assumed to be thermal emission
    # and the temperature whose black body emission that best matches that of
    # T1, is derived. The emissivity is assumed to be constant with wavelength
    # equal to 1- I/F at wavelength A" 
    # "Also, because lunar
    # reflectances in the 2 to 3 mm spectral region are generally
    # lower than 0.5 (but not always), emissivities are usually
    # greater than 0.5 (we assume e = 1 − Ro′ ). For the lowest‐
    # reflectance areas, emissivity is highest. A worst case scenario
    # would be where Ro′ = e = 0.5 and where thermal emission
    # could cancel the absorption caused the reflected component
    # with spectral features."  
    emissivity_const = 1.0 - iof[iA]

    # get temperature w black body emission close to T1 
    T_K1 = fit_temperature(
        T1=T1,
        emissivity=emissivity_const,
        wavelength_c_um=wl[iC],
        solar_spec_at_c=solar_spec[iC],
        rs_sq=1.0,   
    )

    # print(f"T1:{T1}, emissivity:{emissivity_const}, wl:{wl[iC]}, solarspec:{solar_spec[iC]}, T_K1:{T_K1}")
    if T_K1 is None:
        return iof, 0.0, 5

    # STEP 5: Subtract derived thermal emission from the observed IOF, giving a new IOF1 as 
    # a function of wavelength - a single value for all wavelengths or modified by wavelength
    # based on temp? idk? I think for this emissivity is supposed to be constant across 
    # wavelengths. 
    emissivity_vec_1 = np.full_like(wl, emissivity_const)
    thermal_1 = thermal_emission_spectrum(wl, T_K1, emissivity_vec_1, solar_spec, rs_sq=1.0)
    IOF1 = iof_1au - thermal_1 # also called R1 in Clark?
    # "if the computed reflectance is greater than 0.6 then 0.6 is used as the value 
    # for computed reflectance"
    # IOF1 = np.clip(IOF1, a_min=None, a_max=0.6) 

    # STEP 6: "R1 is then corrected for incidence angle and phase angle effects = IOF 1c
    # (as of the writing of this paper, phase angle corrections for the 2 to 3-µm
    # wavelength region for the lunar surface are not yet available)." 
    # min cos value should be 0.05 
    IOF1c = IOF1 / cos_incidence # IOF1c is called R1c in Clark
    # "if the computed reflectance is greater than 0.6 then 0.6 is used as the value 
    # for computed reflectance"
    # IOF1c = np.clip(IOF1c, a_min=None, a_max=0.6) 
    
    # STEP 7: "The wavelength dependent emissivity is then computed from e = 1 –
    # IOF1c."
    # this is based on Kirchhoff’s law (ε = 1-I/F).
    emissivity_vec_2 = 1.0 - IOF1c

    # STEP 8: "Next a new projection using IOF 1 is made from wavelengths D and E
    # (Table 2.6) to a new I/F at wavelength C, and the difference, T2,
    # computed." 
    # BEKAH NOTE: DPSIS says use IOF1 but Clark says use R1c which corresponds to IOF1c
    # so we use IOF1c bc I think that makes more sense? 
    iof_proj_C2 = linear_project(wl, IOF1c, iD, iE, iC)
    # "if the computed reflectance is greater than 0.6 then 0.6 is used as the value 
    # for computed reflectance"
    iof_proj_C2 = np.clip(iof_proj_C2, a_min=None, a_max=0.6) 
    
    T2 = IOF1c[iC] - iof_proj_C2
    #T2 = IOF1[iC] - iof_proj_C2

    # STEP 9: "A new temperature (T) is derived from the thermal difference (T2) by
    # computing a new black body that includes the wavelength dependent
    # emissivity estimate, e, and cosine correction (and once available, a phase
    # correction)."
    # so here emissivity should be wavlength dependant perhaps? 
    T_K2 = fit_temperature(
        T1=T2,
        emissivity=emissivity_vec_2[iC],#emissivity_vec_2[iC],
        wavelength_c_um=wl[iC],
        solar_spec_at_c=solar_spec[iC],
        rs_sq=1.0,
    )
    
    # if T_K2 fit failed then best estimate is IOF1 with T_K1.
    if T_K2 is None:
        return IOF1, T_K1, 0

    # STEP 10: "The thermal emission, ro2eBo(T)/Fsun, is computed from this 2nd iteration
    # estimate and subtracted from the original I/F spectrum, producing a new
    # estimate, IOF2c."
    # BEKAH COMMENT: So should I not have done this before? I'm confused. 
    thermal_2 = thermal_emission_spectrum(wl, T_K2, emissivity_vec_2, solar_spec, rs_sq=1.0)
    IOF2c = ( iof_1au - thermal_2  ) /  cos_incidence
    # "if the computed reflectance is greater than 0.6 then 0.6 is used as the value 
    # for computed reflectance"
    # IOF2c = np.clip(IOF2c, a_min=None, a_max=0.6) 
    
    # STEP 11: DPSIS says: "If the derived temperature difference from the first and secon
    # iteration is less 2 K, the solution is complete. Otherwise, a third iteration is
    # computed by going back to step 6 and substituting IOF2c for IOF1c."

    # Clark says: "If the derived temperatures from the first and
    # second iteration are less the 2 K, the solution is complete.
    # Otherwise, a third iteration was computed by going back to
    # step 6 and substituting R2c for R1c."

    # Bekah says: technically the DPSIS and Clark are saying two different things: diff
    # is less than 2 K or the temps themselves are less than 2 K. But I think Clark must 
    # mean diff between them. 
    
    temp_diff = abs(T_K2 - T_K1)
    if temp_diff < 2:
        return IOF2c, T_K2, 1

    # STEP 11: DPSIS says: "If the derived temperature difference from the first and second
    # iteration is less 2 K, the solution is complete. Otherwise, a third iteration is
    # computed by going back to step 6 and substituting IOF2c for IOF1c." 

    # Clark says: "Otherwise, a third iteration was computed by going back to
    # step 6 and substituting R2c for R1c."

    # Bekah says: Well Clark and DPSIS have different step 6s bc DPSIS has an extra step 1. 
    # Because Clark happened first I assume Clark's 6 is correct, which would be:
    # "The wavelength‐dependent emissivity is then computed from e = 1 − R1c." INSTEAD 
    # of doing the incidence angle correction twice. 

    # STEP 6/7 RND 3 
    emissivity_vec_3 = 1.0 - IOF2c

    # STEP 8 RND 3
    iof_proj_C3 = linear_project(wl, IOF2c, iD, iE, iC)
    # "if the computed reflectance is greater than 0.6 then 0.6 is used as the value 
    # for computed reflectance"
    iof_proj_C3 = np.clip(iof_proj_C3, a_min=None, a_max=0.6) 
    
    T3 = IOF2c[iC] - iof_proj_C3
    #T3 = iof[iC] - iof_proj_C3


    if T3 <= 0:
        return IOF2c, T_K2, 2

    # STEP 9 RND 3 - should the cosine correction be used in another way than the IOF itself? 
    T_K3 = fit_temperature(
        T1=T3,
        emissivity=emissivity_vec_3[iA], #emissivity_vec_3[iC],
        wavelength_c_um=wl[iC],
        solar_spec_at_c=solar_spec[iC],
        rs_sq=1.0,
    )
    if T_K3 is None:
        return IOF2c, T_K2, 3

    # STEP 10 RND 3: subtract THIRD estimate from the ORIGINAL or from the SECOND? 
    thermal_3 = thermal_emission_spectrum(wl, T_K3, emissivity_vec_3, solar_spec, rs_sq=1.0)
    #IOF3c = iof_1au - thermal_3   
    # could be: 
    IOF3c = IOF2c - thermal_3   
    
    return IOF3c, T_K3, 4


def thermal_removal(
    image: np.ndarray,
    wavelengths_um: np.ndarray,
    solar_spec: np.ndarray,
    obs_path: Path,
    d: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Apply iterative thermal removal to a full image cube.

    obs bands (0-indexed in FITS):
        1 aka 0. to-sun azimuth angle (decimal degrees, clockwise from local north) 
        2 aka 1. to-sun zenith angle (decimal degrees, zero at zenith) 
        3 aka 2. to-sensor azimuth angle (decimal degrees, clockwise from local north) 
        4 aka 3. to-sensor zenith angle (decimal degrees, zero at zenith) 
        5 aka 4. observation phase angle (decimaldegrees, in plane of to-sun and to-sensor rays) 
        6 aka 5. to-sun path length (decimal au with scene mean subtracted) 
        7 aka 6. to-sensor path length (decimal meters) 
        8 aka 7. surface slope from DEM (decimal degrees, zero at horizontal) 
        9 aka 8. surface aspect from DEM (decimal degrees, clockwise from local north) 
        10 aka 9. local cosine i (unitless, cosine of angle between to-sun and local DEM facet normal vectors)
    """
    n_frames, n_bands, n_samples = image.shape
    wl = wavelengths_um
    corrected = image.copy()

    # derived temps per pixel 
    temperature_map = np.zeros((n_frames, n_samples), dtype=np.float32)
    # flag map just for checking what step of iterative thermal removal it stopped on
    flag_map = np.zeros((n_frames, n_samples), dtype=np.uint8)

    
    obs = fits.open(obs_path)[0].data

    # STEP 1: remove only the solar distance correction from L2 step 1 to input data
    image = image / d**2
    
    # STEP 6 Pre-cursor: we will need the local incidence angle for step 6 
    # calculate local incidence angle wrt topography, as described in photometric 
    # correction step: 
    # i_topo= incidence angle in degrees as supplied by the equation: 
    # i=acos(cos(obs[1])*cos(obs[7])+sin(obs[1])*sin(obs[7])*cos((obs[0]-
    # obs[8]))). 
    i_topo = np.degrees(np.arccos(
         np.cos(np.radians(obs[1])) * np.cos(np.radians(obs[7]))
         + np.sin(np.radians(obs[1])) * np.sin(np.radians(obs[7]))
           * np.cos(np.radians(obs[0]) - np.radians(obs[8])))
    )
    # If the resulting incidence angle is greater than or equal to 85.0
    # degrees then set i_topo to 85.0 before proceeding with the photometry
    # correction... could not clip it here? maybe they didn't do it for thermal
    # correction? 
    i_topo = np.clip(i_topo, 0.0, 85.0)

    # "We limit the I/F cosine correction and the computed reflectance to avoid
    # errors due to lower spatial resolution to topography and registration
    # topography to prevent very high computed reflectances. If
    # cosine(incidence) is less than 0.05 then 0.05 is used as the value for
    # cosine(incidence)".
    cos_i_topo = np.cos(np.radians(i_topo)) # could avoid the conversion to degrees above
    fits.writeto("cos_i_topo.fits", cos_i_topo, overwrite=True)
    cos_i_topo = np.clip(cos_i_topo, a_min =0.05, a_max=None)
    
    #STEP 6 Pre-cursor: we will need the local incidence angle for step 6 
    # we can use obs[9] instead? 
    # BEKAH: This is an alternative possible value for cosine incidence angle w local topo- when you diff 
    # the method above with using obs[9] (below), the max diff is .002, most pixels significantly lower diff 
    # than that. So I don't think this is a big difference maker in the algorithm. 
    # cos_i_topo = np.clip(obs[9], a_min=0.05, a_max=None)

    del obs

    # get indices of bands closest to wavelengths used for projecting spectra
    iA = nearest_band(wavelengths_um, WL_A)
    iB = nearest_band(wavelengths_um, WL_B)
    iC = nearest_band(wavelengths_um, WL_C)
    iD = nearest_band(wavelengths_um, WL_D)
    iE = nearest_band(wavelengths_um, WL_E)

    # STEPS 2-11: go frame by frame to run iterative thermal removal 
    for f in range(n_frames):
        for s in range(n_samples):
            corr, T_K, flag = remove_thermal_single_spectrum(
                iof_spectrum=image[f, :, s],
                solar_spec=solar_spec,
                cos_incidence=cos_i_topo[f, s],
                d=d,
                wl=wavelengths_um,
                iA=iA, 
                iB=iB, 
                iC=iC, 
                iD=iD, 
                iE=iE,
            )
            corrected[f, :, s] = corr
            temperature_map[f, s] = T_K
            flag_map[f, s] = flag

    
    # return step 1 distance correction
    corrected = corrected * d**2 
    
    fits.writeto("temp_flag_map.fits", flag_map, overwrite=True)
    return corrected, temperature_map


def load_solar_spec_for_thermal(sol_spec_path: Path) -> tuple[np.ndarray, np.ndarray]:
    sol = pd.read_fwf(sol_spec_path, names=["wavelength_nm", "sol_irrad"])
    wavelengths_um = sol["wavelength_nm"].values / 1000.0
    solar_iof      = sol["sol_irrad"].values #/ np.pi  # do I need this or not? removing this improved 
                                                       # how close I was to their values. and the image is multiplied by pi in step 1
    return wavelengths_um, solar_iof


def run_thermal_removal_step(
    image: np.ndarray,
    moonager,
) -> tuple[np.ndarray, np.ndarray]:
    wavelengths_um, solar_iof = load_solar_spec_for_thermal(moonager.sol_spec_path)
    corrected, temp_map = thermal_removal(
        image=image,
        wavelengths_um=wavelengths_um,
        solar_spec=solar_iof,
        obs_path=moonager.l1b_obs_path,
        d=moonager.solar_distance,
    )
    return corrected, temp_map


### Photometric Correction

L2s4(λ) = L2s3(λ) * { XL_norm(i_topo,e_topo,α) * Falpha_norm(α, λ) }\
where\
• λ is the wavelength (global or target); all bands.

i_topo= incidence angle in degrees as supplied by the equation:\
i=acos(cos(obs[1])*cos(obs[7])+sin(obs[1])*sin(obs[7])*cos((obs[0]-obs[8])))\
If the resulting incidence angle is greater than or equal to 85.0 degrees then set i_topo to 85.0 before proceeding with the photometry
correction.

e_topo= emission angle in degrees as supplied by the equation:\
e=acos(cos(obs[3])*cos(obs[7])+sin(obs[3])*sin(obs[7])*cos((obs[2]-obs[8])))

If the resulting emission angle is greater than or equal to 85.0 degrees then set e_topo to 85.0 before proceeding with the photometry
correction.\
OBS[0]= “per pixel to-sun azimuth” band in *OBS.IMG\
OBS[1]= “per pixel to-sun zenith” band in *OBS.IMG\
OBS[2]= “per pixel to-M3 azimuth” band in *OBS.IMG\
OBS[3]= “per pixel to-M3 zenith” band in *OBS.IMG\
OBS[7]= “per pixel facet slope” band in *OBS.IMG\
OBS[8]= “per pixel facet aspect” band in *OBS.IMG

α = phase angle in degrees as supplied by the “per pixel phase” band in *OBS.IMG

#### Photometric Limb Darkening Factor, XL(α), for Global or Target Modes
XL_norm(i_topo,e_topo,α) is normalized to 30º phase; equal to XL(30,0,30) /XL(i_topo,e_topo,α) where:\
XL(i_topo,e_topo,α) = (cos(i_topo) / (cos(e_topo) + cos(i_topo))), a simple Lommel-Seeliger model

#### Falpha_norm(α, λ) correction, for Global or Target Modes
The actual Falpha_norm(α, λ) factor to be applied, is linearly interpreted in α from a look-up table Falpha(α, λ) (global or target, supplied by the science team) of correction factors dependent on α and λ.

Falpha_norm(α, λ) is normalized to 30º phase, Falpha(30, λ) / Falpha(α, λ)

In [ ]:
def load_f_alpha_factors(f_alpha_path: Path, phase_angle: np.ndarray) -> np.ndarray:
    """
    Interpolates Falpha_norm(α, λ) = Falpha(30, λ) / Falpha(α, λ)
    from the lookup table, per pixel.
    """
    from scipy.interpolate import interp1d

    falpha = pd.read_fwf(f_alpha_path)
    
    falpha = falpha.iloc[:100].copy()
    falpha = falpha.set_index('Unnamed: 0')
    falpha.index = falpha.index.astype(float)

    table_angles = falpha.index.values
    table_values = falpha.values

    interp = interp1d(
        table_angles,
        table_values,
        axis=0,
    )

    f_alpha_at_30 = interp(30.0)

    input_shape = phase_angle.shape
    alpha_flat  = phase_angle.ravel()

    f_alpha_at_pixels = interp(alpha_flat)
    f_alpha_norm      = f_alpha_at_30[np.newaxis, :] / f_alpha_at_pixels
    f_alpha_norm      = f_alpha_norm.reshape(*input_shape, 85)

    return f_alpha_norm

def photometric_correction(
        image: np.ndarray,
        f_alpha_path: Path,
        obs_path: Path
) -> np.ndarray:
    """
    Photometric correction using local topography.

    L2s4(λ) = L2s3(λ) * { XL_norm(i_topo,e_topo,α) * Falpha_norm(α, λ) }

    We use geometry info from the l1b obs image to calculate the photometric
    limb darkening factor, Xl.
    The ten 'bands' (axis 1) of the obs file are the following:
    1. to-sun azimuth angle (decimal degrees, clockwise from local north)
    2. to-sun zenith angle (decimal degrees, zero at zenith)
    3. to-sensor azimuth angle (decimal degrees, clockwise from local north)
    4. to-sensor zenith angle (decimal degrees, zero at zenith)
    5. observation phase angle (decimal degrees, in plane of to-sun and
    to-sensor rays)
    6. to-sun path length (decimal au with scene mean subtracted)
    7. to-sensor path length (decimal meters)
    8. surface slope from DEM (decimal degrees, zero at horizontal)
    9. surface aspect from DEM (decimal degrees, clockwise from local north)
    10. local cosine i (unitless, cosine of angle between to-sun and local
    DEM facet normal vectors)
    """
    obs = fits.open(obs_path)[0].data

    f_alpha_norm = load_f_alpha_factors(f_alpha_path, obs[4])

    # we have to convert to radians for np cos etc 
    
    i_topo = np.arccos(
        np.cos(np.radians(obs[1])) * np.cos(np.radians(obs[7]))
        + np.sin(np.radians(obs[1])) * np.sin(np.radians(obs[7]))
          * np.cos(np.radians(obs[0]) - np.radians(obs[8])))

    i_topo = np.degrees(i_topo)
    i_topo[i_topo >= 85.0] = 85.0

    e_topo = np.arccos(
        np.cos(np.radians(obs[3])) * np.cos(np.radians(obs[7]))
        + np.sin(np.radians(obs[3])) * np.sin(np.radians(obs[7]))
          * np.cos(np.radians(obs[2]) - np.radians(obs[8])))

    e_topo = np.degrees(e_topo)
    e_topo[e_topo >= 85.0] = 85.0

    x_l_topo = np.cos(np.radians(i_topo)) / (
        np.cos(np.radians(e_topo)) + np.cos(np.radians(i_topo))
    )
    x_l_30_0_30 = np.cos(np.radians(30)) / (
        np.cos(np.radians(0)) + np.cos(np.radians(30))
    )

    x_l_norm = x_l_30_0_30 / x_l_topo

    del e_topo
    del i_topo
    del obs
    del x_l_topo
    del x_l_30_0_30

    phot_correction = x_l_norm[:, :, np.newaxis] * f_alpha_norm
    image = image * phot_correction.transpose(0, 2, 1)

    return image

### Ground Truth Correction (I have it turned off right now) 

The delivery in the PDS for L2 doesn't have this applied.

L2s5(λ) = L2s4(λ) * gGT(λ) + oGT(λ)\
where\
• λ is the wavelength (global or target)\
• L2s4(λ) is the 3-dimensional, photometrically corrected, reflectance image file
produced by Step 4.\
• gGT(λ) is a Global or Target File of derived gains for a ground truth correction…\
• oGT (λ) is a Global or Target File of derived offsets for a ground truth correction…\
o For this PDS delivery, oGT(λ) is set to zero (no offset).

In [ ]:
def ground_truth_correction(image: np.ndarray, ground_truth_path: Path):
    """
    Ground truth correction was not applied to L2 data in the PDS but they
    provided the cal files.

    L2s5(λ) = L2s4(λ) * gGT(λ) + oGT(λ)

    Here, oGT(λ) is 0.
    """
    import pandas as pd

    ground_truth = pd.read_fwf(ground_truth_path,
                               names=['channel', 'wavelength',
                                      'corr_factor', 'corr_offset'])

    image *= ground_truth['corr_factor'].values[np.newaxis, :, np.newaxis]

    # image += ground_truth['corr_offset'].values[np.newaxis, :, np.newaxis]

    return image

### Bits and bobs for running the whole L2 pipeline 


In [ ]:

def pull_metadata(obs_id: str, cal_dir: Optional[str] = None) -> dict:
    """
    Read metadata scraped from L1B RDN HDRs for the eclipse + some quality
    checks done by hand on flag maps and darks.
    """
    if cal_dir is None:
        cal_dir = CAL_DIR
    if not isinstance(cal_dir, Path):
        cal_dir = Path(cal_dir)

    metadata = pd.read_csv(cal_dir / "obs_cal_info.csv")
    metadata = metadata[metadata['obs_id'] == obs_id.upper()]

    if len(metadata) > 1:
        metadata = metadata.loc[
            metadata['version'].str.extract(r'(\d+)')[0].astype(int).idxmax()]
        return metadata.to_dict()
    return metadata.iloc[0].to_dict()


def check_observation(obs_id: str, local_root: Optional[str] = None):
    """
    Check that the input observation string matches the expected format & load
    info we have on it from the latest version of its L1B files
    (ie V03 not V01). Most of this info was scraped from L1B RDN HDRs in the
    JPL PDS files.

    Some files were never processed to L1B for whatever reason: they are dark
    obs and therefore should not be processed in this way, or they had issues
    (extremely hot etc.). These will return an error and empty dataframe at
    this time.

    Basic obs format: M3MYYYYMMDDTHHMMSS, ex: m3g20090720t214000
    """
    import re

    obs_warn: list[str] = []
    obs_error: list[str] = []

    obs_id_pattern = r'^m3[gt]\d{8}t\d{6}$'
    if not bool(re.match(obs_id_pattern, obs_id, re.IGNORECASE)):
        obs_error.append(f"This is not a valid obs ID: {obs_id}")
        return obs_warn, obs_error, {}

    metadata = pull_metadata(obs_id, local_root)

    if len(metadata) == 0:
        obs_error.append(f"No metadata found for observation {obs_id}.")
        return obs_warn, obs_error

    if metadata['bad_dark']:
        obs_warn.append(f"This has been flagged as a bad dark obs, "
                        f"{metadata['dark_signal_id']}.")
    if metadata['bad_bde']:
        obs_warn.append(f"This has been flagged as a bad BDE map, "
                        f"{metadata['bad_detector_map_id']}.")

    if not metadata['flat_in_usgs']:
        obs_warn.append(f"The obs-based flat for this obs is not in the PDS, "
                        f"{metadata['flat_field_id']}.")
    if not metadata['bde_in_usgs']:
        obs_warn.append(f"The flag map for this obs is not in the PDS, "
                        f"{metadata['bad_detector_map_id']}.")
    if not metadata['dark_in_usgs']:
        obs_warn.append(f"The dark obs for this obs is not in the PDS, "
                        f"{metadata['dark_signal_id']}.")

    if metadata['bde_flag_coverage'] * 100 > 10:
        obs_warn.append(f"{metadata['bde_flag_coverage'] * 100}% of the "
                        f"mission-derived bad element map is flagged for this "
                        f"observation.")

    flat_diff = abs(metadata['obs_temperature'] - metadata['flat_field_temp'])
    if flat_diff > 3:
        obs_warn.append(f"This obs and its flat field obs have a temp "
                        f"difference greater than 3 Kelvin, "
                        f"{flat_diff} K.")

    dark_diff = abs(metadata['obs_temperature'] - metadata['dark_signal_temp'])
    if dark_diff > 3:
        obs_warn.append(f"This obs and its dark have a temp difference greater"
                        f" than 3 Kelvin, "
                        f"{dark_diff} K.")

    return obs_warn, obs_error, metadata


class PipeManager:

    def __init__(self,
                 obs_id: str,
                 local_root: str,
                 metadata: dict,
                 save_steps: bool = False,
                 verbose: bool = True,
                 ):
        self.save_steps = save_steps
        self.verbose    = verbose
        self.local_root = Path(local_root)

        self.obs_id      = obs_id
        self.mode        = metadata['obs_type']
        self.obs_period  = metadata['obs_period']
        self.date        = metadata['obs_date']
        self.time        = metadata['obs_time']
        self.obs_temp    = metadata['obs_temperature']
        self.obs_beta_angle = metadata['obs_beta_angle']

        ### specific to the observation I've been testing! m3g20090108t044645
        self.solar_distance =  0.985107922514

        self.date_time = datetime.fromisoformat(f"{self.date}T{self.time}")
        self.temp_des  = (
            "cold" if any(
                datetime.fromisoformat(s) <= self.date_time < datetime.fromisoformat(e)
                for s, e in [
                    ("2009-01-19T00:00:00", "2009-02-15T00:00:00"),
                    ("2009-04-15T00:00:00", "2009-04-28T00:00:00"),
                    ("2009-07-12T00:00:00", "2009-08-17T00:00:00"),
                ]
            ) else "warm" if any(
                datetime.fromisoformat(s) <= self.date_time < datetime.fromisoformat(e)
                for s, e in [
                    ("2008-11-18T00:00:00", "2009-01-19T00:00:00"),
                    ("2009-05-13T00:00:00", "2009-05-17T00:00:00"),
                    ("2009-05-20T00:00:00", "2009-07-10T00:00:00"),
                ]
            ) else None
        )

        self.dark_temp    = metadata['dark_signal_temp']
        self.dark_id      = metadata['dark_signal_id'].lower()
        self.flag_id      = metadata['bad_detector_map_id'].lower()
        self.obs_flat_id  = metadata['flat_field_id'].lower()
        self.l0_obs_path  = self.local_root / f'{self.obs_id}_l0.fits'
        self.dark_path    = self.local_root / f'{self.dark_id}_l0.fits'
        self.obs_flat_path = self.local_root / f'{self.obs_flat_id}_ff.fits'
        self.flag_path    = self.local_root / f'{self.flag_id}_bde.fits'
        self.ssc_path     = self.local_root / f'{self.obs_id}_ssc.txt'

        if self.mode.upper() == 'T':
            self.lab_flat_path = Path(CAL_DIR) / 'lab_flat_field_target.fits'
            self.rdn_gain_path = Path(CAL_DIR) / 'm3t20070912_rdn_gain.tab'
            self.rdn_spc_path  = Path(CAL_DIR) / 'm3t20070912_rdn_spc.tab'
            self.rdn_cal_path  = Path(CAL_DIR) / 'm3t20081118_rdn_cal.tab'
        elif self.mode.upper() == 'G':
            self.lab_flat_path = Path(CAL_DIR) / 'lab_flat_field_global.fits'
            self.rdn_gain_path = Path(CAL_DIR) / 'm3g20081211_rdn_gain.tab'
            self.rdn_spc_path  = Path(CAL_DIR) / 'm3g20081211_rdn_spc.tab'
            self.rdn_cal_path  = Path(CAL_DIR) / 'm3g20081118_rdn_cal.tab'

        self.ghost_corr_factor = .0048

        if self.mode.upper() == 'T':
            self.l0_samples    = 640
            self.l0_channels   = 260
            self.l1b_samples   = 608
            self.l1b_channels  = 256
            self.omitted_channels = [0, 1, 2, 3]
            self.dark_cols     = [1, 2, 3, 4, 5, 6, 7, 636, 637, 638, 639]
            self.vignetted_cols_left  = [8, 9, 10, 11, 12, 13, 14]
            self.vignetted_cols_right = [627, 628, 629, 630, 631, 632, 633, 634, 635]
            self.left_col_cutoff  = 17
            self.right_col_cutoff = 625
            self.read_out_cols    = [0, 160, 320, 480]
            self.filter_seam_rows = [40, 41, 115]
            self.degraded_channels = [0, 1, 2, 3, 4, 5, 6, 7]

        elif self.mode.upper() == 'G':
            self.l0_samples   = 320
            self.l0_channels  = 86
            self.l1b_samples  = 304
            self.l1b_channels = 85
            self.omitted_channels = [0]
            self.dark_cols    = [1, 2, 3, 318, 319]
            self.vignetted_cols_left  = [4, 5, 6]
            self.vignetted_cols_right = [314, 315, 316, 317]
            self.vignetted_cols = [4, 5, 6, 314, 315, 316, 317]
            self.left_col_cutoff  = 9
            self.right_col_cutoff = 313
            self.read_out_cols    = [0, 80, 160, 240]
            self.filter_seam_rows = [12, 49]
            self.degraded_channels = [0, 1]

        self.l1b_rdn_path = self.local_root / f'{self.obs_id}_l1b_rdn.fits'
        self.l1b_obs_path = self.local_root / f'{self.obs_id}_l1b_obs.fits'
        self.l1b_tim_path = self.local_root / f'{self.obs_id}_l1b_tim.tab'
        self.l1b_loc_path = self.local_root / f'{self.obs_id}_loc.fits'

        if self.mode.upper() == 'T':
            self.sol_spec_path       = Path(CAL_DIR) / 'm3t20110224_rfl_solar_spec.tab'
            self.old_f_alpha_hil_path = Path(CAL_DIR) / 'm3t20111109_rfl_f_alpha_hil.tab'
            self.new_f_alpha_hil_path = Path(CAL_DIR) / 'm3t20120120_rfl_f_alpha_hil.tab'
            if self.temp_des == 'warm':
                self.grnd_truth_path = Path(CAL_DIR) / 'm3t20111117_rfl_grnd_tru_2.tab'
                self.stat_pol_path   = Path(CAL_DIR) / 'm3t20111020_rfl_stat_pol_2.tab'
            elif self.temp_des == 'cold':
                self.grnd_truth_path = Path(CAL_DIR) / 'm3t20111117_rfl_grnd_tru_1.tab'
                self.stat_pol_path   = Path(CAL_DIR) / 'm3t20111020_rfl_stat_pol_1.tab'
        elif self.mode.upper() == 'G':
            self.sol_spec_path       = Path(CAL_DIR) / 'm3g20110224_rfl_solar_spec.tab'
            self.old_f_alpha_hil_path = Path(CAL_DIR) / 'm3g20111109_rfl_f_alpha_hil.tab'
            self.new_f_alpha_hil_path = Path(CAL_DIR) / 'm3g20120120_rfl_f_alpha_hil.tab'
            if self.temp_des == 'warm':
                self.grnd_truth_path = Path(CAL_DIR) / 'm3g20111117_rfl_grnd_tru_2.tab'
                self.stat_pol_path   = Path(CAL_DIR) / 'm3g20110830_rfl_stat_pol_2.tab'
            elif self.temp_des == 'cold':
                self.grnd_truth_path = Path(CAL_DIR) / 'm3g20111117_rfl_grnd_tru_1.tab'
                self.stat_pol_path   = Path(CAL_DIR) / 'm3g20110830_rfl_stat_pol_1.tab'


def load_fits_into_frame(obs_path: Path) -> np.ndarray:
    """
    Load the obs data and return it in detector POV / frame view where
    axis 0 is frames / lines.
    """
    from astropy.io import fits

    filename = obs_path.name.lower()

    with fits.open(obs_path) as hdul:
        image = hdul[0].data

    if "ff" in filename or "flat" in filename:
        return image.astype(np.float32)

    if "bde" in filename:
        return image.astype(np.uint8)

    return image.transpose(1, 0, 2).astype(np.float32)


def run_l2_mission_pipeline(moonager: PipeManager):
    """
    Pipeline for L1B (radiance) to L2 (reflectance) based on an originalist
    reading of the DPSIS. This version of the pipeline uses multiple products
    from L1B level processing, including topo data etc. You could (at some
    point) use these products from our version of the L1B pipeline, or use the
    L1B products stored by the USGS PDS node (fits files).
    """

    image = load_fits_into_frame(moonager.l1b_rdn_path)

    # (1) I/F Correction
    if moonager.verbose:
        print("Converting radiance to radiance factor (I/F).")
    image = calculate_radiance_factor(
        image=image,
        sol_spec_path=moonager.sol_spec_path, 
        d=moonager.solar_distance
    )

    fits.writeto("if_map.fits", image.transpose(1, 0, 2), overwrite=True)

    # (2) Statistical Polishing
    if moonager.verbose:
        print("Applying statistical polishing factors.")
    image = apply_stat_polishing(
        image=image,
        stat_pol_path=moonager.stat_pol_path
    )

    fits.writeto("stat_pol_map.fits", image.transpose(1, 0, 2), overwrite=True)

    # (3) Iterative Thermal Removal
    if moonager.verbose:
        print("Running iterative thermal removal.")
    image, temp_map = run_thermal_removal_step(image, moonager)

    fits.writeto("temp_map.fits", temp_map, overwrite=True)
    del temp_map

    # (4) Photometric Correction
    if moonager.verbose:
        print("Running photometric correction of entire cube.")
    image = photometric_correction(
        image=image,
        f_alpha_path=moonager.old_f_alpha_hil_path,
        obs_path=moonager.l1b_obs_path
    )

    # (4b) Photometric Correction of only 1489 nm relative to a sphere
    # not implemented yet

    # (5) Ground Truth Correction (Optional)
    # if moonager.verbose:
    #     print("Applying ground truth correction.")
    # image = ground_truth_correction(
    #     image=image,
    #     ground_truth_path=moonager.grnd_truth_path
    # )

    # (6) Flag Degraded Channels
    if moonager.verbose:
        print("Flagging degraded channels.")
    image[:, moonager.degraded_channels, :] = -999.0

    fits.writeto("l2_test.fits", image, overwrite=True)

    return image


def run_pipe(
    obs_id: str,
    pipe_version: str = 'mission',
    local_root: str = "data",
    save_steps: bool = False,
    verbose: bool = True,
):
    obs_warn, obs_error, metadata = check_observation(obs_id)
    if verbose and len(obs_warn) > 0:
        print("\n".join(obs_warn))
    if len(obs_error) > 0:
        print("\n".join(obs_error))
        print("Bailing out.")
        return f"return code: {';'.join(obs_error)}"

    moonager = PipeManager(
        obs_id=obs_id,
        metadata=metadata,
        local_root=local_root,
        save_steps=save_steps,
        verbose=verbose,
    )

    print(moonager.temp_des)
    return run_l2_mission_pipeline(moonager)


In [ ]:

DATA_ROOT = Path('/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/crop')

OBS_ID = 'm3g20090108t044645'

CAL_DIR = '/home/bekah/m3-pipeline-dev/l0_l1b_l2/cal_data'

result_mission = run_pipe(
    obs_id=OBS_ID,
    pipe_version='mission',
    local_root=str(DATA_ROOT),
    save_steps=False,
    verbose=True
)

del result_mission

### Compare temperature maps 

In [ ]:
### check temperature differences between their temperature map and ours 

from astropy.io import fits

new_path = "/home/bekah/m3-pipeline-dev/notebooks/temp_map.fits"
with fits.open(new_path) as src:
    new_image = src[0].data
    
old_path = "/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/m3g20090108t044645_l2_sup.fits"
with fits.open(old_path) as src:
    old_image = src[0].data[1, 0:1000, :]
    
fits.writeto(
            f"temp_diff.fits", 
            new_image-old_image, 
            overwrite=True
        )

In [ ]:
# from astropy.io import fits

# new_path = "/home/bekah/m3-pipeline-dev/notebooks/cos_i_topo.fits"
# with fits.open(new_path) as src:
#     new_image = src[0].data
    
# old_path = "/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/crop/m3g20090108t044645_l1b_obs.fits"
# with fits.open(old_path) as src:
#     old_image = src[0].data[9] #.data[1, 0:1000, :]
    
# fits.writeto(
#             f"cos_i_topo_diff_with_obs_9.fits", 
#             new_image-old_image, 
#             overwrite=True
#         )